In [1]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
candidate_paths = [
    Path.cwd() / "ds_job_interview_prep" / "initial_plan.json",
    Path.cwd().parent / "ds_job_interview_prep" / "initial_plan.json",
    Path("ds_job_interview_prep/initial_plan.json"),
]
initial_plan_path = next(
    (p for p in candidate_paths if p.is_file()),
    Path("ds_job_interview_prep/initial_plan.json"),
)
if not initial_plan_path.exists():
    raise FileNotFoundError(
        f"Could not find initial_plan.json. Checked: {[str(p) for p in candidate_paths]}"
    )
initial_plan_path

PosixPath('/Users/mayazwang/Files/00_Learning/_GenAI_Proj/AI_Engineering_FT_RAG_Agent_playground/ds_job_interview_prep/initial_plan.json')

In [3]:

initial_plan_path = Path.cwd() / "initial_plan.json"
with initial_plan_path.open("r", encoding="utf-8") as f:
    initial_plan = json.load(f)

revised_plan_path = Path.cwd() / "revised_plan.json"
with revised_plan_path.open("r", encoding="utf-8") as f:
    revised_plan = json.load(f)

final_plan_path = Path.cwd() / "final_plan.json"
with final_plan_path.open("r", encoding="utf-8") as f:
    final_plan = json.load(f)

In [4]:

def get_plan_text(value, day="Day 1"):
    """Extract the requested day's section from common API response structures."""
    if not re.fullmatch(r"Day\s+\d+", day.strip(), re.IGNORECASE):
        raise ValueError("day must look like 'Day 1' or 'Day 2'")

    def to_text(item):
        if isinstance(item, str):
            return item
        if isinstance(item, dict):
            for key in ("content", "text", "output_text"):
                if key in item:
                    text = to_text(item[key])
                    if text:
                        return text
            return "\n".join(filter(None, (to_text(v) for v in item.values())))
        if isinstance(item, (list, tuple)):
            return "\n".join(filter(None, (to_text(v) for v in item)))
        return ""

    text = to_text(value)
    heading_pattern = re.compile(
        r"(?im)^\s{0,3}(?:#{1,6}\s*)?(?:\d+[.)]\s*)?(?:\*\*)?(Day\s+\d+)\b.*$"
    )
    headings = list(heading_pattern.finditer(text))
    requested_day = re.sub(r"\s+", " ", day.strip()).casefold()

    for index, heading in enumerate(headings):
        found_day = re.sub(r"\s+", " ", heading.group(1)).casefold()
        if found_day == requested_day:
            end = headings[index + 1].start() if index + 1 < len(headings) else len(text)
            return text[heading.start():end].strip()

    raise ValueError(f"{day!r} was not found in the response")

In [ ]:
initial_plan_text = get_plan_text(initial_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(initial_plan_text))

1. **Day 1 — Translate Zillow’s mandate into a measurement strategy**
   - **Daily Objective**
     - Define what success means for Connections Platform (CP), distinguish measurement from instrumentation problems, and establish the causal questions behind its metrics.
     - **Role analysis:** Prioritize causal inference, LLM/agent evaluation, event instrumentation, platform-as-a-product measurement, and executive communication. Python and SQL should support these capabilities—not generic algorithm drills.
     - **Source transparency:** Live web search is unavailable in this chat. The linked authoritative resources are recommended readings, not freshly verified sources. Interview formats below are preparation targets inferred from the job description, not confirmed Zillow interview rounds.
     - **Schedule for every day: exactly 4 hours:** Python **40 minutes**; SQL **35 minutes**; review **45 minutes**; causal case study **90 minutes**; additional exercises and debrief **30 minutes**. Total: **56 hours**.
     - **Notebook specification:** Outline one notebook per day, beginning with `day01_measurement_strategy.ipynb`. Each notebook should contain objectives, source links, imports/data loading, challenge cells, review markdown, case analysis, solution notes/tests, and a final decision memo. No actual notebook files are created here.
     - **Shared conventions:** Use pandas, NumPy, DuckDB, SciPy, statsmodels, and scikit-learn; add specialized packages when indicated. Register DataFrames with DuckDB for SQL exercises. Cache public data and record download dates. Explicitly label simulated treatments, outcomes, and instrumentation.

   - **Python and SQL Challenges**
     - **Python:** Implement a metric function returning numerator, denominator, rate, and Wilson confidence interval. Handle empty cohorts and missing outcomes without silently treating missingness as failure.
       - **Solution key:** Compute the denominator from the prespecified eligible population. For observed binary outcomes, use `from statsmodels.stats.proportion import proportion_confint`, followed by `proportion_confint(successes, observed_n, method="wilson")`. Report missingness separately.
     - **SQL:** On the bank dataset below, calculate subscription rates and sample sizes by contact channel.
       - **Reference segment:** `SELECT contact, COUNT(*) AS n, AVG(CASE WHEN y='yes' THEN 1.0 ELSE 0.0 END) AS subscription_rate FROM bank GROUP BY contact;`
       - **Key:** These are descriptive channel comparisons—not causal channel effects.

   - **Review Topics**
     - Define the chain: eligible consumer → meaningful conversation → qualified connection → partner response → downstream transaction.
     - Draft a metric tree containing:
       - **Primary business outcome:** completed, qualified connections per eligible lead.
       - **Leading indicators:** extraction correctness, successful task completion, partner acceptance.
       - **Lagging indicators:** downstream transactions and incremental contribution margin.
       - **Guardrails:** unwanted outreach, opt-outs, complaints, fair access, latency, and cost.
     - Define treatment, assignment unit, exposure, estimand, outcome horizon, and interference.
     - Explain why adoption, message volume, and agent invocation counts are insufficient evidence of value.

   - **Case Study — Does a contact channel improve lead conversion?**
     - **Public dataset:** UCI Bank Marketing. It is a contact/conversion analogue, not Zillow data; subscription is not a home transaction.
     - **Dataset access code:**
       - `import pandas as pd; from io import BytesIO; from zipfile import ZipFile; from urllib.request import urlopen`
       - `z = ZipFile(BytesIO(urlopen("https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip").read()))`
       - `df = pd.read_csv(z.open("bank-full.csv"), sep=";")`
     - **Tasks:** Draw a DAG for contact channel and subscription. Define the desired causal effect. Identify which denominators and confounders are missing.
     - **Brief solution key:** Channel assignment was not randomized. Customer characteristics, campaign decisions, and prior engagement can confound the comparison. Call duration is not a safe pretreatment adjustment variable. The dataset covers contacted customers, not the entire eligible lead population.

   - **Additional Resources/Exercises**
     - Read [Zillow’s technology publications](https://www.zillow.com/tech/) and [Zillow Research](https://www.zillow.com/research/) for company and housing-market context.
     - Use the opening chapters of [Causal Inference: What If](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) for estimands and identification.
     - **Conclusion/deliverable:** A one-page CP measurement charter with five metric definitions, two causal questions, and three instrumentation gaps.

In [6]:
revised_plan_text = get_plan_text(revised_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(revised_plan_text))

### Day 1 — Translate Zillow's mandate into a measurement strategy
`day01_measurement_strategy.ipynb`

**Objective:** Define success for Connections Platform (CP), separate measurement problems from instrumentation problems, and learn the business well enough to tie metrics to revenue.

**Python:** Implement a metric function returning numerator, denominator, rate, and Wilson interval. Handle empty cohorts and missing outcomes without treating missingness as failure.
- Key: Take the denominator from the prespecified eligible population. Use `proportion_confint(successes, observed_n, method="wilson")` from `statsmodels.stats.proportion`. Report missingness separately.

**SQL:** Subscription rate and sample size by contact channel.
- `SELECT contact, COUNT(*) AS n, AVG(CASE WHEN y='yes' THEN 1.0 ELSE 0.0 END) AS subscription_rate FROM bank GROUP BY contact;`
- Key: These are descriptive comparisons, not causal channel effects.

**Review:**
- The chain: eligible consumer → meaningful conversation → qualified connection → partner response → downstream transaction.
- First-draft metric tree: primary outcome, leading indicators, lagging indicators, guardrails (refined on Day 2).
- Treatment, assignment unit, exposure, estimand, outcome horizon, interference.
- Why adoption, message volume, and agent invocation counts are not evidence of value.
- **[NEW] Business primer:**
  - How Zillow earns revenue from connections (agent advertising and success-fee programs, rentals, mortgages).
  - The partner-facing tools.
  - The lag between a connection and a closed transaction.
  - Source this from the latest shareholder letter and 10-K, not from memory.
- **[NEW] Three customers:** consumers, partners (agents, loan officers), and internal developer teams. Write one sentence on what "better" means for each.
- **[NEW] Constraints:** fair-housing obligations and consent rules for calls and texts. Know they exist and that legal partners own the interpretation.
- **[NEW] Method map v0:** draft it from what you know today.

**Case study — Does a contact channel improve lead conversion?**
- Dataset: UCI Bank Marketing (a contact/conversion analogue, not Zillow data).
```python
import pandas as pd; from io import BytesIO; from zipfile import ZipFile; from urllib.request import urlopen
z = ZipFile(BytesIO(urlopen("https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip").read()))
df = pd.read_csv(z.open("bank-full.csv"), sep=";")
```
- Tasks: Draw a DAG for channel and subscription. Define the causal effect. List missing denominators and confounders.
- Key:
  - Channel was not randomized. Customer traits, campaign decisions, and prior engagement confound the comparison.
  - Call duration is not a safe pretreatment variable.
  - The data covers contacted customers, not the eligible population.

**Out-loud drill [NEW]:** "How would you spend your first 90 days measuring CP?" (five minutes, recorded). Then list eight story titles mapped to the themes on Day 14.

**Resources:** [Zillow tech blog](https://www.zillow.com/tech/); [Zillow Research](https://www.zillow.com/research/); **[NEW]** [Zillow investor relations](https://investors.zillowgroup.com/); [Causal Inference: What If](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/), opening chapters.

**Deliverable:** One-page CP measurement charter (five metric definitions, two causal questions, three instrumentation gaps), method map v0, story list.

---

In [7]:
final_plan_text = get_plan_text(final_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(final_plan_text))

### Day 1 — Zillow’s Two-Sided Marketplace Mandate & Measurement Strategy `[UPGRADED]`
`notebooks/day01_measurement_strategy.ipynb`

**Objective:** Translate Zillow’s Connections Platform (CP) and CoreTech mandate into a rigorous measurement charter anchored in real-estate marketplace economics, separating measurement problems from instrumentation problems.

**Python:** Implement a reusable rate-metric calculator in `src/cp_eval_metrics/stats.py` returning `eligible_n`, `observed_n`, `missing_rate`, `numerator`, `point_estimate`, and Wilson score interval (`proportion_confint(..., method="wilson")`). Write `pytest` tests verifying empty cohorts, zero-numerator bounds, and strict separation of missing outcomes from failures.

**SQL `[UPGRADED]`:** On UCI Bank Marketing (`bank-full.csv`), build a cohort summary stratified by contact channel and prior campaign contact status (`pdays != 999` / `previous > 0`), computing conversion rate, share of total volume, and post-treatment leakage diagnostics (`duration`).
```sql
SELECT
    contact,
    CASE WHEN previous > 0 THEN 'prior_touch' ELSE 'cold_lead' END AS lead_history,
    COUNT(*) AS contacted_n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_volume,
    AVG(CASE WHEN y = 'yes' THEN 1.0 ELSE 0.0 END) AS conv_rate,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY duration) AS median_call_sec_post_treatment
FROM bank
GROUP BY 1, 2
ORDER BY 1, 2;
```
* **Key:** Conditioning on or controlling for `duration` introduces severe collider/mediator bias because call length happens *after* channel assignment and correlates directly with conversion intent.

**Review:**
* **Zillow CP Business & Revenue Primer `[UPGRADED]`:** Read Zillow Group’s latest 10-K / Shareholder Letter. Map how CP connects to monetization:
  * *Residential (Premier Agent & Zillow Flex):* Upfront market-based advertising vs. Flex success fees (paid only when a connection closes a home transaction months later).
  * *Rentals & Mortgages (Zillow Home Loans):* Cost-per-lease/lead dynamics and loan origination pull-through.
  * **Why "More Connections $\neq$ More Revenue":** Partner agents have finite time. Sending unvetted, AI-inflated leads congests partner queues, degrades speed-to-lead on high-intent buyers, and lowers Flex close rates.
* **Three Customer Lenses:** Consumers (fast, accurate, unpressured answers), Partners (high-intent, well-summarized handoffs that convert), Internal Product/Eng Teams (reliable shared APIs, low latency, clear schemas).
* **Regulatory & Trust Guardrails `[UPGRADED]`:** Fair Housing Act (anti-steering, equal service quality across neighborhoods/demographics) and TCPA consent (opt-in/opt-out state across SMS and automated voice). Legal defines policy, *but Data Science must operationalize deterministic compliance metrics and zero-tolerance eval gates*.
* **Method Map v0:** Draft your baseline decision matrix across A/B, cluster/switchback, DiD, synthetic control, ITS, interleaving, and encouragement designs.

**Case Study — Why Observational Channel Metrics Mislead Executive Decisions**
* **Dataset:** UCI Bank Marketing (`https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip`).
* **Tasks:**
  1. Draw the causal DAG linking Consumer Intent (unobserved), Channel Selection (`cellular` vs. `telephone` vs. `unknown`), Conversation Engagement (`duration`), and Subscription (`y`).
  2. Identify three missing denominators (e.g., eligible uncontacted visitors, failed dial/SMS deliveries, opted-out leads).
  3. Quantify how naive regression controlling for `duration` distorts the apparent channel gap compared to pre-treatment covariates only.

**Out-Loud Drill:** *"Walk me through your first 90 days as Principal DS for Connections Platform and CoreTech."* (5 minutes, recorded). Draft your 8 Story Bank titles.

**Deliverable:** 1-page CP & CoreTech Measurement Charter, Method Map v0, Story Bank index, and initial `stats.py` + unit tests.

---